# Task 2
This serves as a template which will guide you through the implementation of this task. It is advised to first read the whole template and get a sense of the overall structure of the code before trying to fill in any of the TODO gaps.
This is the jupyter notebook version of the template. For the python file version, please refer to the file `template_solution.py`.

First, we import necessary libraries:

In [1]:
import numpy as np
import pandas as pd
# Add any other imports you need here
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, RBF, Matern, RationalQuadratic

# test
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.gaussian_process.kernels import WhiteKernel

# Data Loading
TODO: Perform data preprocessing, imputation and extract X_train, y_train and X_test
(and potentially change initialization of variables to accomodate how you deal with non-numeric data)

In [2]:
"""
This loads the training and test data, preprocesses it, removes the NaN
values and interpolates the missing data using imputation

Parameters
----------
Compute
----------
X_train: matrix of floats, training input with features
y_train: array of floats, training output with labels
X_test: matrix of floats: dim = (100, ?), test input with features
"""
# Load training data
train_df = pd.read_csv("train.csv")
    
print("Training data:")
print("Shape:", train_df.shape)
print(train_df.head(2))
print('\n')
    
# Load test data
test_df = pd.read_csv("test.csv")

print("Test data:")
print(test_df.shape)
print(test_df.head(2))

# Dummy initialization of the X_train, X_test and y_train   
# TODO: Depending on how you deal with the non-numeric data, you may want to 
# modify/ignore the initialization of these variables
"""
X_train = np.zeros_like(train_df.drop(['price_CHF'],axis=1))
y_train = np.zeros_like(train_df['price_CHF'])
X_test = np.zeros_like(test_df)
"""

# TODO: Perform data preprocessing, imputation and extract X_train, y_train and X_test

# remove rows that done have a CHF price so we dont train on made up data
train_df = train_df.dropna(subset=['price_CHF'])

# preprocessing using pandas get_dummies function
train_df = pd.get_dummies(train_df, columns=['season'])
test_df = pd.get_dummies(test_df, columns=['season'])

# proper initialization of X_train, X_test and y_train
X_train = train_df.drop(['price_CHF'],axis=1)
#X_train_season = X_train['season']              # TEMP
#X_train = X_train.drop(['season'],axis=1)       # TEMP
y_train = train_df['price_CHF']
X_test = test_df
#X_test_season = X_test['season']                # TEMP
#X_test = X_test.drop(['season'],axis=1)         # TEMP

#print("train_df", X_train.head(5))
#print("test_df", X_test.head(5))

# imputation time using the median
#imp = SimpleImputer(missing_values=np.nan, strategy='median')

imp = IterativeImputer(max_iter=10, random_state=0)


X_train = imp.fit_transform(X_train)
X_test = imp.transform(X_test)
#print(X_train)


assert (X_train.shape[1] == X_test.shape[1]) and (X_train.shape[0] == y_train.shape[0]) and (X_test.shape[0] == 100), "Invalid data shape"

Training data:
Shape: (900, 11)
   season  price_AUS  price_CHF  price_CZE  price_GER  price_ESP  price_FRA  \
0  spring  -3.348808        NaN  -3.597534  -4.102160  -2.201652  -2.806995   
1  summer  -3.421345  -1.455502  -3.597649  -3.675204        NaN  -2.440406   

   price_UK  price_ITA  price_POL  price_SVK  
0       NaN   -3.61728  -2.758448        NaN  
1 -2.379524        NaN        NaN   -3.72506  


Test data:
(100, 10)
   season  price_AUS  price_CZE  price_GER  price_ESP  price_FRA  price_UK  \
0  spring  -1.504285  -1.632302  -2.347618        NaN        NaN -3.437325   
1  summer  -1.779837  -1.750216  -2.407555  -1.875685        NaN       NaN   

   price_ITA  price_POL  price_SVK  
0  -3.505886  -2.042408        NaN  
1  -3.528359  -2.131659  -2.911154  


# Modeling and Prediction
TODO: Define the model and fit it using training data. Then, use test data to make predictions

In [3]:
"""
This defines the model, fits training data and then does the prediction
with the test data 

Parameters
----------
X_train: matrix of floats, training input with 10 features
y_train: array of floats, training output
X_test: matrix of floats: dim = (100, ?), test input with 10 features

Compute
----------
y_test: array of floats: dim = (100,), predictions on test set
"""
class Model(object):
    def __init__(self):
        super().__init__()
        self._x_train = None
        self._y_train = None
        # store the best regressor here
        self.best_reg = None

    def fit(self, X_train: np.ndarray, y_train: np.ndarray):
        #TODO: Define the model and fit it using (X_train, y_train)
        self._x_train = X_train
        self._y_train = y_train

        # We put all the kernels as candidates and then check which one gives the best result
        candidates = [
            DotProduct(),   # linear kernel
            RBF(length_scale=1.0),  # squared exponential kernel
            Matern(length_scale=1.0, nu=1.5),
            RationalQuadratic(length_scale=1.0, alpha=0.1)
        ]

        best_lml = -np.inf

        # Loop over the candidates to see which kernel is the best
        """
        for kernel in candidates:
            gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5)
            gpr.fit(X_train, y_train)

            lml = gpr.log_marginal_likelihood(gpr.kernel_.theta)

            if lml > best_lml:
                best_lml = lml
                self.best_reg = gpr
        print(self.best_reg)
        """
        gpr = GaussianProcessRegressor(kernel=RBF(), n_restarts_optimizer=3, normalize_y=True)
        gpr.fit(X_train, y_train)
        self.best_reg = gpr


    def predict(self, X_test: np.ndarray) -> np.ndarray:
        #TODO: Use the model to make predictions y_pred using test data X_test
        # y_pred=np.zeros(X_test.shape[0])
        y_pred = self.best_reg.predict(X_test)
        assert y_pred.shape == (X_test.shape[0],), "Invalid data shape"
        return y_pred

In [4]:
model = Model()
# Use this function to fit the model
model.fit(X_train=X_train, y_train=y_train)
# Use this function for inference
y_pred = model.predict(X_test)
print(y_pred)

[ 5.92604696e+00  4.04190752e+00  3.54307309e+00  4.91541224e+00
  2.74393220e+00  2.92589701e+00  2.17059354e+00  2.04948212e+00
  9.94246031e-01  7.06351141e-01  1.02789020e+00 -4.55112778e-01
 -7.44689430e-01 -9.79566108e-01 -4.91523424e-02 -1.88213388e+00
  2.19729434e-01 -3.35524977e+00 -1.83650539e-01 -1.36665061e+00
 -2.58603990e+00 -2.99007267e+00 -2.03495685e+00  2.14740760e+00
 -1.53828754e+00 -2.23427168e+00 -5.84411814e-01 -8.21166780e-01
 -4.47613456e-01 -4.53941243e-01 -9.89459570e-01 -2.11726622e+00
 -7.96681078e-01 -1.08515533e+00 -5.52100809e-01 -8.63185957e-01
 -9.51967562e-01 -1.09276340e+00 -1.97388008e+00 -1.10497624e+00
 -2.69425323e+00 -1.02541288e+00 -1.67856731e+00 -8.65058143e-01
 -1.84566555e+00 -7.20244578e-01 -1.32349873e+00 -1.72469263e+00
 -1.44999782e+00 -1.25117079e+00 -1.42279000e+00 -1.19356633e+00
 -1.82748902e+00 -1.18018699e+00 -1.69546188e+00 -7.97868705e-01
 -8.77678851e-01 -1.17550474e+00 -1.20581660e+00 -5.91259160e-01
 -7.91060788e-01  5.60725

# Saving Results
You don't have to change this

In [5]:
dt = pd.DataFrame(y_pred) 
dt.columns = ['price_CHF']
dt.to_csv('results.csv', index=False)
print("\nResults file successfully generated!")


Results file successfully generated!
